In [ ]:
!pip install ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import LabelEncoder, StandardScaler

# fetch dataset
student_performance = fetch_ucirepo(id=320)

# data (as pandas dataframes)
X = student_performance.data.features
y = student_performance.data.targets

df = pd.concat([X,y], axis=1)
df = df.dropna()

def grade_category(g):
  if g >= 15:
    return "high"
  elif g >= 10:
    return "medium"
  else:
    return "low"

df['target'] = df['G3'].apply(grade_category)

df = df.drop(columns=['G1', 'G2', 'G3'])

X = df.drop(columns=['target'])
y = df['target']

for col in X.select_dtypes(include='object').columns:
  X[col] = LabelEncoder().fit_transform(X[col])

y = LabelEncoder().fit_transform(y)

X_scaled = StandardScaler().fit_transform(X)

print("Shape:", X_scaled.shape)

#--------------------------------------------------

from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedKFold, cross_val_score

models = {
    'Linear Classifier': Perceptron(max_iter=1000, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Gaussian NB': GaussianNB(),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(64,), max_iter=500, random_state=42),
}

rkf = RepeatedKFold(n_splits=10, n_repeats=100, random_state=42)

print("----- Part 2 Results -----")
for name, model in models.items():
  scores = cross_val_score(model, X_scaled, y, cv=rkf,scoring='accuracy', n_jobs=-1)
  print(f"{name:20s} mean={scores.mean():.4f} std={scores.std():.4f}")

#----------Logistic Regression----------
from sklearn.feature_selection import SequentialFeatureSelector
cv = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

model = LogisticRegression(max_iter=1000, random_state=42)

sfs = SequentialFeatureSelector(
      estimator=model,
      n_features_to_select=10,
      direction='forward',
      cv=cv,
      n_jobs=-1
)

sfs.fit(X_scaled, y)
selected = np.array(X.columns)[sfs.get_support()]
scores = cross_val_score(
    model,
    X_scaled[:, sfs.get_support()],
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Logistic Regression")
print("Selected Features:", list(selected))
print(f"Performance: mean={scores.mean():.4f}, std={scores.std():.4f}")

#----------Perceptron----------
model = Perceptron(max_iter=1000, random_state=42)

sfs = SequentialFeatureSelector(
      estimator=model,
      n_features_to_select=10,
      direction='forward',
      cv=cv,
      n_jobs=-1
)
sfs.fit(X_scaled, y)
selected = np.array(X.columns)[sfs.get_support()]
scores = cross_val_score(
    model,
    X_scaled[:, sfs.get_support()],
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Perceptron")
print("Selected Features:", list(selected))
print(f"Performance: mean={scores.mean():.4f}, std={scores.std():.4f}")

#----------KNN----------
model = KNeighborsClassifier(n_neighbors=5)

sfs = SequentialFeatureSelector(
      estimator=model,
      n_features_to_select=10,
      direction='forward',
      cv=cv,
      n_jobs=-1
)
sfs.fit(X_scaled, y)
selected = np.array(X.columns)[sfs.get_support()]
scores = cross_val_score(
    model,
    X_scaled[:, sfs.get_support()],
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("KNN")
print("Selected Features:", list(selected))
print(f"Performance: mean={scores.mean():.4f}, std={scores.std():.4f}")

#----------Gaussian NB----------
model = GaussianNB()

sfs = SequentialFeatureSelector(
      estimator=model,
      n_features_to_select=10,
      direction='forward',
      cv=cv,
      n_jobs=-1
)
sfs.fit(X_scaled, y)
selected = np.array(X.columns)[sfs.get_support()]
scores = cross_val_score(
    model,
    X_scaled[:, sfs.get_support()],
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Gaussian NB")
print("Selected Features:", list(selected))
print(f"Performance: mean={scores.mean():.4f}, std={scores.std():.4f}")

#----------Neural Network----------
model = MLPClassifier(hidden_layer_sizes=(64, ), max_iter=300, random_state=42)

sfs = SequentialFeatureSelector(
      estimator=model,
      n_features_to_select=10,
      direction='forward',
      cv=cv,
      n_jobs=-1
)
sfs.fit(X_scaled, y)
selected = np.array(X.columns)[sfs.get_support()]
scores = cross_val_score(
    model,
    X_scaled[:, sfs.get_support()],
    y,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

print("Neural Network")
print("Selected Features:", list(selected))
print(f"Performance: mean={scores.mean():.4f}, std={scores.std():.4f}")

Shape: (649, 30)
----- Part 2 Results -----
Linear Classifier    mean=0.5674 std=0.0626
Logistic Regression  mean=0.6341 std=0.0584
KNN                  mean=0.5910 std=0.0569
Gaussian NB          mean=0.3720 std=0.0567
